In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import KMNIST
from torch.utils.data import DataLoader
import numpy as np

# 1. random seed for reproducibility
torch.manual_seed(42)

In [2]:
# Device configuration (Use GPU if available, otherwise fallback to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# 2. Hyperparameters 
EPOCHS     = 20  # Minimum 20 epochs required by the project guidelines
BATCH_SIZE = 64  # Mini-batch size of 64
LR         = 1e-3 # Adam default learning rate (0.001)

In [4]:
# 3. Load and Standardize the KMNIST Dataset 
# First, load the raw data with a basic transform to calculate the mean and standard deviation
base_transform = transforms.Compose([transforms.ToTensor()])
temp_trainset = KMNIST(root='./pt_data', train=True, download=True, transform=base_transform)
temp_loader = DataLoader(temp_trainset, batch_size=len(temp_trainset), shuffle=False)

data = next(iter(temp_loader))
mean = data[0].mean().item()
stddev = data[0].std().item()

# Final transform using the calculated mean and standard deviation for normalization
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, stddev)
])

# Final datasets and data loaders
trainset = KMNIST(root='./pt_data', train=True, download=True, transform=transform)
testset  = KMNIST(root='./pt_data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
testloader  = DataLoader(testset,  batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 18.2M/18.2M [00:51<00:00, 353kB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 100kB/s]
100%|██████████| 3.04M/3.04M [00:09<00:00, 337kB/s] 
100%|██████████| 5.12k/5.12k [00:00<00:00, 374kB/s]


In [6]:
# 4. Model Architecture 
class KMNISTFeedForwardNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        # Define the linear layers
        self.hidden1 = nn.Linear(784, 128) # Input (784) -> Hidden 1 (128 neurons)
        self.hidden2 = nn.Linear(128, 64)  # Hidden 1 (128) -> Hidden 2 (64 neurons)
        self.output  = nn.Linear(64, 10)   # Hidden 2 (64) -> Output (10 classes)
        
        # Weight Initialization scheme (Following your reference code pattern)
        # Kaiming (He) initialization for the ReLU hidden layers
        nn.init.kaiming_normal_(self.hidden1.weight)
        nn.init.constant_(self.hidden1.bias, 0.0)
        nn.init.kaiming_normal_(self.hidden2.weight)
        nn.init.constant_(self.hidden2.bias, 0.0)
        # Xavier (Glorot) initialization for the output layer (no activation)
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.constant_(self.output.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Flatten the 2D image (28x28) into a 1D vector (784)
        x = torch.flatten(x, start_dim=1)
        # Apply ReLU activation function to hidden layers as requested
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        # Return raw logits; softmax will be internally applied by the loss function
        return self.output(x)

# Create the model instance and move it to the configured device
model = KMNISTFeedForwardNet().to(device)

In [7]:
# 5. Loss Function and ADAM Optimizer Setup
loss_function = nn.CrossEntropyLoss()

# Adam Optimizer with default hyperparameters 
optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=LR, 
    betas=(0.9, 0.999), 
    eps=1e-8
)

In [8]:
# 6. Training and Evaluation Function (Single-function approach from your reference code)
def run_epoch(model, loader, criterion, optimizer, device, training):
    # Set model to training or evaluation mode
    model.train() if training else model.eval()

    total_loss    = 0.0
    total_correct = 0
    total_samples = 0

    # Enable gradients only during training, disable during testing to save memory
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            # Backward pass and Optimization step (Only during training phase)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Accumulate evaluation metrics
            _, predicted   = torch.max(outputs, dim=1)
            total_correct += (predicted == targets).sum().item()
            total_samples += targets.size(0)
            total_loss    += loss.item()

    # Return average loss and average accuracy for the epoch
    return total_loss / len(loader), total_correct / total_samples

In [9]:
# 7. Final Training and Testing Loop
print("--- Training and Validation Loop Started ---")
for epoch in range(EPOCHS):
    # Run training phase
    train_loss, train_acc = run_epoch(
        model, trainloader, loss_function, optimizer, device, training=True
    )
    # Run testing (validation) phase
    test_loss, test_acc = run_epoch(
        model, testloader, loss_function, optimizer=None, device=device, training=False
    )
    
    # Print metrics (* 100 converts decimal accuracy to percentage format)
    print(f'Epoch {epoch+1}/{EPOCHS} -> Loss: {train_loss:.4f} - Acc: {train_acc*100:.2f}% | Test Loss: {test_loss:.4f} - Test Acc: {test_acc*100:.2f}%')

--- Training and Validation Loop Started ---
Epoch 1/20 -> Loss: 0.3619 - Acc: 88.74% | Test Loss: 0.4792 - Test Acc: 85.41%
Epoch 2/20 -> Loss: 0.1677 - Acc: 94.92% | Test Loss: 0.4556 - Test Acc: 87.11%
Epoch 3/20 -> Loss: 0.1140 - Acc: 96.46% | Test Loss: 0.4565 - Test Acc: 87.49%
Epoch 4/20 -> Loss: 0.0879 - Acc: 97.27% | Test Loss: 0.4838 - Test Acc: 87.73%
Epoch 5/20 -> Loss: 0.0683 - Acc: 97.80% | Test Loss: 0.5489 - Test Acc: 87.15%
Epoch 6/20 -> Loss: 0.0534 - Acc: 98.25% | Test Loss: 0.4944 - Test Acc: 88.85%
Epoch 7/20 -> Loss: 0.0433 - Acc: 98.60% | Test Loss: 0.4913 - Test Acc: 89.37%
Epoch 8/20 -> Loss: 0.0374 - Acc: 98.78% | Test Loss: 0.5689 - Test Acc: 88.77%
Epoch 9/20 -> Loss: 0.0341 - Acc: 98.84% | Test Loss: 0.6207 - Test Acc: 88.70%
Epoch 10/20 -> Loss: 0.0285 - Acc: 99.07% | Test Loss: 0.6336 - Test Acc: 89.06%
Epoch 11/20 -> Loss: 0.0241 - Acc: 99.20% | Test Loss: 0.6367 - Test Acc: 88.93%
Epoch 12/20 -> Loss: 0.0265 - Acc: 99.12% | Test Loss: 0.6723 - Test Acc:

### Conclusion

In this experiment, a neural network model was trained on the Kuzushiji-MNIST (KMNIST) dataset using the Adam optimizer with default parameters. The model was trained for 20 epochs, during which the training accuracy steadily increased and reached approximately 99.5%, indicating that the network learned the training data effectively.

On the test dataset, the model achieved a maximum accuracy of about 89.8%, demonstrating good generalization performance. However, while the training loss continued to decrease, the test loss increased during later epochs, suggesting the presence of overfitting. This indicates that the model became increasingly specialized to the training data and its performance on unseen data did not improve significantly after a certain point.

Overall, the results show that the Adam optimizer with default settings provides fast convergence and high classification accuracy on the KMNIST dataset. Nevertheless, techniques such as early stopping, regularization, dropout, or hyperparameter tuning could be applied to further improve the model's generalization and reduce overfitting.